In [ ]:
# Refine School Water Usage Data for 2026-2027 Academic Calendar
This notebook refines `no_leak.csv`, `leak_normal.csv`, and `leak_event.csv` so timestamps align with a US academic calendar and water usage patterns reflect school days, weekends, and breaks.
</VSCode.Cell>
<VSCode.Cell id="2" language="python">
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# File paths
files = {
    'no_leak': 'no_leak.csv',
    'leak_normal': 'leak_normal.csv',
    'leak_event': 'leak_event.csv'
}
</VSCode.Cell>
<VSCode.Cell id="3" language="markdown">
## Load water usage CSV datasets
Read the CSV files into pandas DataFrames for processing.
</VSCode.Cell>
<VSCode.Cell id="4" language="python">
dfs = {name: pd.read_csv(path, parse_dates=['Timestamp']) for name, path in files.items()}
for name, df in dfs.items():
    print(name, len(df), df['Timestamp'].min().date(), df['Timestamp'].max().date())
</VSCode.Cell>
<VSCode.Cell id="5" language="markdown">
## Define academic calendar and break periods
Specify the 2026-2027 academic year timeline and non-school periods.
</VSCode.Cell>
<VSCode.Cell id="6" language="python">
# Academic year boundaries and break periods
academic_start = datetime(2026, 8, 17)
academic_end = datetime(2027, 6, 9)
summer_break_start = datetime(2026, 6, 14)
summer_break_end = datetime(2026, 8, 16)
winter_break_start = datetime(2026, 12, 20)
winter_break_end = datetime(2027, 1, 3)
spring_break_start = datetime(2027, 3, 20)
spring_break_end = datetime(2027, 3, 26)

# Generate full date range for the academic year plus the break start in June 2026
full_dates = pd.date_range(start='2026-01-01', end='2027-06-09', freq='D')

# Identify date types
non_school_dates = set(pd.date_range(summer_break_start, summer_break_end, freq='D')).union(
    pd.date_range(winter_break_start, winter_break_end, freq='D'),
    pd.date_range(spring_break_start, spring_break_end, freq='D')
)

weekends = set([d for d in full_dates if d.weekday() >= 5])

# School days are days in the academic period excluding breaks and weekends
school_days = [d for d in full_dates if academic_start <= d <= academic_end and d.weekday() < 5 and d not in non_school_dates]
print('Total full dates:', len(full_dates))
print('School days:', len(school_days))
print('Non-school dates:', len(non_school_dates))
</VSCode.Cell>
<VSCode.Cell id="7" language="markdown">
## Generate aligned timestamps for 2026-2027 school year
Refine the Timestamp columns for each dataset using the defined academic calendar.
</VSCode.Cell>
<VSCode.Cell id="8" language="python">
# Build an aligned academic calendar list of timestamped entries for the dataset
aligned_dates = list(full_dates)

# Ensure each dataset uses the full date range, trimming or expanding as needed
for name, df in dfs.items():
    if len(df) > len(aligned_dates):
        df = df.iloc[:len(aligned_dates)].copy()
    else:
        padded = pd.DataFrame({
            'Timestamp': aligned_dates[len(df):],
            'Flow_Rate_LPM': np.nan,
            'Avg_Pressure_PSI': np.nan,
            'Occupancy_Status': 'After_Hours'
        })
        df = pd.concat([df, padded], ignore_index=True)
    df['Timestamp'] = aligned_dates[:len(df)]
    dfs[name] = df
    print(name, 'after align', len(df), df['Timestamp'].min().date(), df['Timestamp'].max().date())
</VSCode.Cell>
<VSCode.Cell id="9" language="markdown">
## Adjust flow rate and pressure for school days, weekends, and breaks
Modify values to represent realistic usage differences.
</VSCode.Cell>
<VSCode.Cell id="10" language="python">
# Helper functions for realistic scaling
np.random.seed(42)

def fill_usage(df, scale_factor_school, scale_factor_weekend, scale_factor_break, pressure_signal):
    values = []
    pressures = []
    status = []
    for d in df['Timestamp']:
        if d in non_school_dates or d < academic_start or d > academic_end:
            values.append(np.round(np.random.uniform(1.5, 3.0), 1))
            pressures.append(np.round(np.random.uniform(30.0, 35.0), 1))
            status.append('Break')
        elif d.weekday() >= 5:
            values.append(np.round(np.random.uniform(4.5, 7.0) * scale_factor_weekend, 1))
            pressures.append(np.round(np.random.uniform(38.0, 42.0) * pressure_signal, 1))
            status.append('Weekend')
        else:
            values.append(np.round(np.random.uniform(12.0, 18.0) * scale_factor_school, 1))
            pressures.append(np.round(np.random.uniform(48.0, 53.0) * pressure_signal, 1))
            status.append('Class_Hours')
    df['Flow_Rate_LPM'] = values
    df['Avg_Pressure_PSI'] = pressures
    df['Occupancy_Status'] = status
    return df

# Apply patterns separately for each dataset
pattern_params = {
    'no_leak': (1.0, 0.4, 0.2, 1.0),
    'leak_normal': (1.1, 0.45, 0.2, 1.0),
    'leak_event': (1.3, 0.55, 0.25, 1.1)
}

for name, df in dfs.items():
    params = pattern_params[name]
    dfs[name] = fill_usage(df.copy(), *params)
    print(name, 'sample')
    print(dfs[name].head(3))
</VSCode.Cell>
<VSCode.Cell id="11" language="markdown">
## Apply integrity checks and compare distributions
Validate that school days have higher spikes while breaks and weekends remain low.
</VSCode.Cell>
<VSCode.Cell id="12" language="python">
for name, df in dfs.items():
    school = df[df['Occupancy_Status'] == 'Class_Hours']
    weekend = df[df['Occupancy_Status'] == 'Weekend']
    brk = df[df['Occupancy_Status'] == 'Break']
    print(f"\n{name}")
    print('School avg flow:', round(school['Flow_Rate_LPM'].mean(), 1), 'avg pressure:', round(school['Avg_Pressure_PSI'].mean(), 1))
    print('Weekend avg flow:', round(weekend['Flow_Rate_LPM'].mean(), 1), 'avg pressure:', round(weekend['Avg_Pressure_PSI'].mean(), 1))
    print('Break avg flow:', round(brk['Flow_Rate_LPM'].mean(), 1), 'avg pressure:', round(brk['Avg_Pressure_PSI'].mean(), 1))
    print('Counts:', len(school), len(weekend), len(brk))
</VSCode.Cell>
<VSCode.Cell id="13" language="markdown">
## Save updated CSV files
Write the refined datasets back to the original CSV files.
</VSCode.Cell>
<VSCode.Cell id="14" language="python">
for name, path in files.items():
    dfs[name].to_csv(path, index=False)
    print('Saved', path)
</VSCode.Cell>
<VSCode.Cell id="15" language="markdown">
## Git shell commands for add, commit, and push
Use these commands in the repository root to stage the updated files and push to `main`.
</VSCode.Cell>
<VSCode.Cell id="16" language="markdown">
```bash
git add no_leak.csv leak_normal.csv leak_event.csv
git commit -m 'Update: Refine academic calendar data for HydroSentinel-AI'
git push origin main
```
